# Deteccion de Outliers y Anomalias _(version detallada)_

_Identificar y gestionar valores extremos con metodos estadisticos y de ML._

**Modulo 1 — Feature Engineering & Feature Stores** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

![Outlier Detection](assets/header.png)

> 📖 **Version detallada** — Incluye comentarios linea a linea en cada bloque de codigo
> y ampliaciones teoricas en las secciones de contexto.  
> Para una lectura mas rapida, consulta la version original `02_outlier_detection.ipynb`.

## Introduccion

Un **outlier** es una observacion que se aleja significativamente del patron general de los
datos. En el dataset de Ames Housing, las casas con `GrLivArea > 4000 sqft` son outliers
famosos: son propiedades atipicas que pueden distorsionar los modelos lineales y las
metricas basadas en distancias.

**¿Por que importa detectar outliers?**

- Los modelos lineales (Ridge, Lasso) son sensibles a valores extremos porque el MCO
  minimiza el error **cuadratico** — un outlier extremo puede arrastrar toda la recta.
- Los algoritmos de distancia (k-NN, DBSCAN, KNNImputer) se ven distorsionados si
  la escala no es uniforme.
- Las metricas estadisticas (media, varianza) quedan sesgadas.

**No todos los outliers son errores.** A veces son las observaciones mas interesantes
(fraudes, fallos de maquinaria, casas de lujo). La decision de eliminar, transformar o
marcar un outlier depende del dominio y del objetivo.

### Metodos cubiertos en este notebook

| Metodo | Tipo | Robusto a outliers multiples | Capta multivariable |
|---|---|---|---|
| **z-score** | Estadistico | No (usa media/std) | No |
| **IQR / Tukey** | Estadistico | Si (usa Q1/Q3) | No |
| **MAD** | Estadistico | Si (usa mediana) | No |
| **Isolation Forest** | ML | Si | Si |
| **DBSCAN** | ML (densidad) | Si | Si |

## Setup y datos

Usamos el dataset **Ames Housing**. El helper `load_housing()` busca el CSV en rutas
candidatas y, si no lo encuentra, genera un DataFrame sintetico con las columnas clave.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(0)


def load_housing():
    """Localiza housing_train.csv buscando rutas candidatas (offline-friendly).

    Busca, en orden: la variable de entorno HOUSING_CSV y varias rutas relativas
    al notebook. Si no encuentra el CSV, construye un pequeno DataFrame sintetico
    con las columnas clave para que el notebook siga corriendo.
    """
    import os

    # Lista de rutas candidatas donde buscar el CSV.
    candidates = []
    # Si el usuario define HOUSING_CSV como variable de entorno, la usamos primero.
    if os.environ.get("HOUSING_CSV"):
        candidates.append(os.environ["HOUSING_CSV"])
    # Rutas relativas tipicas segun la estructura del repositorio.
    candidates += [
        os.path.join("..", "..", "data", "housing_train.csv"),
        os.path.join("..", "..", "..", "data", "housing_train.csv"),
        os.path.join("data", "housing_train.csv"),
    ]
    # Intentamos cada ruta candidata.
    for path in candidates:
        if path and os.path.exists(path):
            df = pd.read_csv(path)
            print(f"Dataset Ames Housing cargado desde: {path}  shape={df.shape}")
            return df

    # Fallback: creamos un DataFrame sintetico con las columnas clave.
    print("CSV no encontrado: usando datos sinteticos con las columnas clave.")
    rng = np.random.default_rng(0)
    n = 400
    overall_qual = rng.integers(3, 10, n)
    gr_liv_area  = (overall_qual * 180 + rng.normal(400, 250, n)).clip(400, 5500)
    # Injectamos outliers claros para que los metodos de deteccion funcionen bien.
    gr_liv_area[-5:] = [4600, 4800, 5000, 4700, 4900]
    sale = (overall_qual * 22000 + gr_liv_area * 55 + rng.normal(0, 25000, n)).clip(40000, 600000)
    df = pd.DataFrame({
        "Id": np.arange(1, n + 1),
        "OverallQual": overall_qual,
        "GrLivArea": gr_liv_area.round().astype(int),
        "TotalBsmtSF": (gr_liv_area * 0.6 + rng.normal(0, 150, n)).clip(0, 3000).round().astype(int),
        "LotArea": rng.gamma(2.0, 5000, n).clip(1300, 60000).round().astype(int),
        "YearBuilt": rng.integers(1900, 2010, n),
        "LotFrontage": np.where(rng.random(n) < 0.18, np.nan,
                                rng.integers(30, 120, n)).astype(float),
        "SalePrice": sale.round().astype(int),
    })
    return df


df = load_housing()
print("Columnas disponibles:", list(df.columns))
df[["Id", "GrLivArea", "SalePrice", "OverallQual"]].head()

In [ ]:
# Exploracion inicial: distribucion de las variables clave.
# GrLivArea (superficie habitable) es donde suelen aparecer los outliers famosos de Ames.
price = df["SalePrice"].dropna()
area  = df["GrLivArea"].dropna()

print(f"SalePrice: n={len(price):,}  media={price.mean():.0f}  mediana={price.median():.0f}  std={price.std():.0f}")
print(f"GrLivArea: n={len(area):,}  media={area.mean():.0f}  mediana={area.median():.0f}  std={area.std():.0f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

# Histograma de GrLivArea: si hay outliers a la derecha, veremos una cola larga.
axes[0].hist(area, bins=40, color="#4c78a8", edgecolor="white")
axes[0].set_title("Distribucion de GrLivArea")
axes[0].set_xlabel("GrLivArea (sqft)")
axes[0].set_ylabel("conteo")

# Scatter GrLivArea vs SalePrice: los outliers suelen aparecer como puntos aislados
# o puntos que no siguen la tendencia lineal general.
idx = area.index.intersection(price.index)
axes[1].scatter(area[idx], price[idx], s=10, alpha=0.4, color="#4c78a8")
# Marcamos el umbral tipico de outlier en Ames (>4000 sqft).
axes[1].axvline(4000, color="#e45756", lw=2, ls="--", label="umbral 4000 sqft")
axes[1].set_title("GrLivArea vs SalePrice")
axes[1].set_xlabel("GrLivArea")
axes[1].set_ylabel("SalePrice")
axes[1].legend()

plt.suptitle("Exploracion inicial: buscando valores extremos", fontweight="bold")
plt.tight_layout()
plt.show()

## 1. Metodos estadisticos de deteccion de outliers

Los metodos estadisticos trabajan sobre una sola variable a la vez (univariante).
Son rapidos, interpretables y no requieren entrenamiento.

### 1a. z-score

$$z_i = \frac{x_i - \mu}{\sigma}, \qquad \text{outlier si } |z_i| > 3$$

**Limitacion:** $\mu$ y $\sigma$ se ven arrastrados por los propios outliers. Con datos
muy sesgados, el z-score de los outliers queda artificialmente bajo (*efecto de
enmascaramiento*). Para distribuciones simetricas sin colas extremas funciona bien.

### 1b. IQR / vallas de Tukey

$$\text{Valla inf} = Q_1 - 1.5 \cdot \text{IQR}, \qquad \text{Valla sup} = Q_3 + 1.5 \cdot \text{IQR}$$

**Ventaja:** usa $Q_1, Q_3$ (robustos a outliers). Es el metodo del boxplot.

### 1c. MAD y z-score modificado

$$\text{MAD} = \text{mediana}(|x_i - \tilde{x}|), \qquad M_i = \frac{0.6745(x_i - \tilde{x})}{\text{MAD}}, \qquad \text{outlier si } |M_i| > 3.5$$

El factor $0.6745$ hace que, para una normal, $\text{MAD} \approx \sigma$. Es el metodo mas
robusto de los tres: ni la media ni la desviacion estandar entran en la formula.

In [ ]:
# --- Metodo 1a: z-score ---
# Trabajamos sobre GrLivArea sin NaN.
area_clean = df["GrLivArea"].dropna()

# mu: media de la columna (SENSIBLE a outliers).
mu = area_clean.mean()
# sigma: desviacion estandar (tambien SENSIBLE a outliers).
sigma = area_clean.std()

# z: puntuacion estandar de cada valor. Mide cuantas desviaciones
# estandar esta cada punto respecto a la media.
z = (area_clean - mu) / sigma

# Umbral clasico: outlier si |z| > 3.
# En una distribucion normal, el 99.7% de los datos caen en [-3, 3].
outliers_z = area_clean[np.abs(z) > 3]

print(f"z-score — mu={mu:.0f}  sigma={sigma:.0f}")
print(f"Outliers (|z|>3): {len(outliers_z)} casas")
if len(outliers_z) > 0:
    print(f"Valores: {sorted(outliers_z.values)[::-1][:10]}")

In [ ]:
# --- Metodo 1b: IQR / vallas de Tukey ---
# Q1 y Q3 son robustos: no les afectan los valores extremos.
Q1 = area_clean.quantile(0.25)   # percentil 25
Q3 = area_clean.quantile(0.75)   # percentil 75
IQR = Q3 - Q1                    # rango intercuartil (50% central de los datos)

# Las vallas de Tukey definen el rango "normal".
# 1.5 * IQR es el factor clasico (propuesto por Tukey en 1977).
valla_inf = Q1 - 1.5 * IQR
valla_sup = Q3 + 1.5 * IQR

# Identificamos puntos fuera de las vallas.
outliers_iqr = area_clean[(area_clean < valla_inf) | (area_clean > valla_sup)]

print(f"IQR — Q1={Q1:.0f}  Q3={Q3:.0f}  IQR={IQR:.0f}")
print(f"Valla inferior: {valla_inf:.0f}  |  Valla superior: {valla_sup:.0f}")
print(f"Outliers (fuera de vallas): {len(outliers_iqr)} casas")
if len(outliers_iqr) > 0:
    print(f"Valores superiores: {sorted(outliers_iqr[outliers_iqr > valla_sup].values)[::-1][:10]}")

In [ ]:
# --- Metodo 1c: MAD y z-score modificado ---
# La mediana es mucho mas robusta que la media ante valores extremos.
mediana = area_clean.median()

# MAD: mediana de las desviaciones absolutas respecto a la mediana.
# Concepto: cuanto se desvia 'tipicamente' un punto de la mediana.
mad = np.median(np.abs(area_clean - mediana))

# Evitamos division por cero si MAD = 0 (distribucion constante).
mad_safe = mad if mad > 0 else 1e-10

# z-score modificado: el factor 0.6745 escala el MAD para que sea
# comparable con la desviacion estandar en una distribucion normal.
# Para una N(mu, sigma): MAD ≈ 0.6745 * sigma, de ahi el factor.
m_mod = 0.6745 * (area_clean - mediana) / mad_safe

# Umbral: 3.5 (equivalente a los 3.0 del z-score estandar).
outliers_mad = area_clean[np.abs(m_mod) > 3.5]

print(f"MAD — mediana={mediana:.0f}  MAD={mad:.0f}")
print(f"Outliers z-mod (|M|>3.5): {len(outliers_mad)} casas")
if len(outliers_mad) > 0:
    print(f"Valores: {sorted(outliers_mad.values)[::-1][:10]}")

print(f"\nResumen comparativo:")
print(f"  z-score (|z|>3)  : {len(outliers_z):3d} outliers")
print(f"  IQR (Tukey)      : {len(outliers_iqr):3d} outliers")
print(f"  MAD (|M|>3.5)    : {len(outliers_mad):3d} outliers")

## 2. Metodos basados en ML

Los metodos estadisticos trabajan variable a variable. Los metodos de ML detectan
outliers en el **espacio multivariable**: una observacion puede ser normal en cada
variable individualmente pero anomala en su combinacion (p.ej. una casa muy grande
a un precio muy bajo).

### 2a. Isolation Forest

**Idea.** Un outlier es facil de "aislar": construye un arbol de decision aleatorio que
parte los datos por valores aleatorios. Un punto anomalo necesita **muy pocas particiones**
para quedar aislado (camino corto desde la raiz). Un punto normal necesita muchas.

**Score de anomalia:** la profundidad promedio de aislamiento en el bosque. Cuanto
mas cerca de la raiz (profundidad menor), mas anomalo.

**Hiperparametro clave:** `contamination` — fraccion esperada de outliers en los datos.
Con `contamination=0.05` le decimos que esperamos ~5% de outliers. Internamente
fija el umbral del score de anomalia.

**Ventajas:** funciona bien en alta dimension, no asume ninguna distribucion, es eficiente
($O(n \log n)$). **Desventaja:** el `contamination` hay que calibrarlo con conocimiento
del dominio.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Seleccionamos las features para la deteccion multivariable.
# Usamos GrLivArea y SalePrice para capturar casas grandes pero baratas (o viceversa).
iso_cols = ["GrLivArea", "SalePrice", "OverallQual", "TotalBsmtSF"]
iso_cols = [c for c in iso_cols if c in df.columns]
X_iso = df[iso_cols].dropna()

# Estandarizamos ANTES de Isolation Forest.
# Aunque IF usa particiones aleatorias (no distancias), la escala puede sesgar
# las particiones hacia las variables de mayor rango.
scaler_iso = StandardScaler()
X_iso_std = scaler_iso.fit_transform(X_iso)

# Instanciamos Isolation Forest.
# contamination=0.05: esperamos que ~5% de las casas sean outliers.
# n_estimators=100: numero de arboles de aislamiento (mas = mas estable).
# random_state=0: semilla para reproducibilidad.
iso = IsolationForest(contamination=0.05, n_estimators=100, random_state=0)

# fit_predict: aprende el modelo Y asigna etiquetas en un solo paso.
# Retorna: 1 = normal, -1 = outlier (anomalia).
labels_iso = iso.fit_predict(X_iso_std)

# score_samples: puntuacion de anomalia (mas negativo = mas anomalo).
# Umbral interno: iso.threshold_ (calculado en fit segun contamination).
scores_iso = iso.score_samples(X_iso_std)

n_outliers = (labels_iso == -1).sum()
print(f"Isolation Forest — features: {iso_cols}")
print(f"Outliers detectados: {n_outliers} ({n_outliers/len(X_iso):.1%} de {len(X_iso)} casas)")
print(f"Umbral de score: {iso.threshold_:.4f}")

# Mostramos las casas mas anomalas segun el score.
X_iso_result = X_iso.copy()
X_iso_result["anomaly_score"] = scores_iso
X_iso_result["is_outlier"] = (labels_iso == -1)
print("\nTop-5 casas mas anomalas (score mas negativo):")
print(X_iso_result.sort_values("anomaly_score").head())

### 2b. DBSCAN — deteccion por densidad

**Idea.** DBSCAN (*Density-Based Spatial Clustering of Applications with Noise*) agrupa
puntos en regiones densas. Los puntos que no pertenecen a ningun cluster (demasiado
aislados) se marcan como **ruido** (etiqueta `-1`), que equivale a outliers.

**Hiperparametros:**
- `eps`: radio maximo de vecindad. Dos puntos son vecinos si su distancia $\leq$ `eps`.
- `min_samples`: minimo de vecinos para que un punto sea **core point** (parte densa).

**Reglas:**
- **Core point:** tiene $\geq$ `min_samples` vecinos en radio `eps`.
- **Border point:** vecino de un core point pero con menos de `min_samples` propios.
- **Noise (outlier):** ni core ni border — etiqueta `-1`.

**Importante:** DBSCAN requiere escalar los datos porque usa distancias euclidianas.
Sin escalado, `GrLivArea` (en miles) domina la distancia sobre `OverallQual` (en unidades).

In [ ]:
from sklearn.cluster import DBSCAN

# Usamos GrLivArea y SalePrice para la deteccion en 2D (mas facil de visualizar).
db_cols = ["GrLivArea", "SalePrice"]
db_cols = [c for c in db_cols if c in df.columns]
X_db = df[db_cols].dropna()

# PASO 1: Escalar. DBSCAN usa distancias euclidianas; sin escalar, GrLivArea
# (en cientos) domina y SalePrice (en miles) queda subestimado.
scaler_db = StandardScaler()
X_db_std = scaler_db.fit_transform(X_db)

# PASO 2: Ajustar DBSCAN.
# eps=0.5: dos puntos son vecinos si su distancia estandarizada <= 0.5.
#          (0.5 desviaciones estandar es una vecindad relativamente estrecha).
# min_samples=5: necesitamos al menos 5 vecinos para ser un punto core.
db = DBSCAN(eps=0.5, min_samples=5)

# fit_predict: ejecuta DBSCAN y asigna etiquetas.
# Retorna: cluster 0, 1, 2, ... para los clusters; -1 para ruido (outliers).
labels_db = db.fit_predict(X_db_std)

# Contamos cuantos puntos son ruido (outliers para DBSCAN).
n_outliers_db = (labels_db == -1).sum()
n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)

print(f"DBSCAN — eps=0.5, min_samples=5")
print(f"Clusters encontrados: {n_clusters}")
print(f"Outliers (ruido, label=-1): {n_outliers_db} ({n_outliers_db/len(X_db):.1%} de {len(X_db)} casas)")

X_db_result = X_db.copy()
X_db_result["cluster"] = labels_db
print("\nDistribucion de etiquetas de cluster:")
print(X_db_result["cluster"].value_counts().sort_index().to_string())

In [ ]:
# Visualizacion comparativa: todos los metodos aplicados a GrLivArea vs SalePrice.
# Cada subplot muestra que puntos marca cada metodo como outlier.
idx_plot = X_db.index
area_p  = df.loc[idx_plot, "GrLivArea"]
price_p = df.loc[idx_plot, "SalePrice"]

# Calculamos los outliers de z-score e IQR en el subconjunto comun.
area_sub  = area_p.dropna()
mu_s, sg_s = area_sub.mean(), area_sub.std()
z_sub = (area_sub - mu_s) / sg_s
out_z_mask = np.abs(z_sub) > 3

Q1s = area_sub.quantile(0.25); Q3s = area_sub.quantile(0.75)
IQRs = Q3s - Q1s
out_iqr_mask = (area_sub < Q1s - 1.5*IQRs) | (area_sub > Q3s + 1.5*IQRs)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

# Subplot 1: z-score + IQR (metodos estadisticos univariantes)
normal_mask = ~out_z_mask & ~out_iqr_mask
axes[0].scatter(area_sub[normal_mask], price_p.loc[area_sub[normal_mask].index],
                s=8, alpha=0.4, color="#4c78a8", label="normal")
axes[0].scatter(area_sub[out_z_mask], price_p.loc[area_sub[out_z_mask].index],
                s=40, color="#e45756", zorder=5, label=f"z-score ({out_z_mask.sum()})")
axes[0].scatter(area_sub[out_iqr_mask], price_p.loc[area_sub[out_iqr_mask].index],
                s=40, marker="^", color="#f58518", zorder=5, label=f"IQR ({out_iqr_mask.sum()})")
axes[0].set_title("Estadisticos univariantes\n(z-score + IQR sobre GrLivArea)")
axes[0].set_xlabel("GrLivArea"); axes[0].set_ylabel("SalePrice"); axes[0].legend(fontsize=8)

# Subplot 2: Isolation Forest (multivariante)
out_iso_mask = (labels_iso == -1)
griv_iso = X_iso["GrLivArea"]
sale_iso = X_iso["SalePrice"]
axes[1].scatter(griv_iso[~out_iso_mask], sale_iso[~out_iso_mask],
                s=8, alpha=0.4, color="#4c78a8", label="normal")
axes[1].scatter(griv_iso[out_iso_mask], sale_iso[out_iso_mask],
                s=40, color="#e45756", zorder=5, label=f"outlier ({out_iso_mask.sum()})")
axes[1].set_title(f"Isolation Forest (multivariante)\ncontamination=5%")
axes[1].set_xlabel("GrLivArea"); axes[1].legend(fontsize=8)

# Subplot 3: DBSCAN (densidad)
out_db_mask = (labels_db == -1)
axes[2].scatter(area_p[~out_db_mask], price_p[~out_db_mask],
                s=8, alpha=0.4, color="#4c78a8", label="cluster")
axes[2].scatter(area_p[out_db_mask], price_p[out_db_mask],
                s=40, color="#e45756", zorder=5, label=f"ruido ({out_db_mask.sum()})")
axes[2].set_title(f"DBSCAN (densidad)\neps=0.5, min_samples=5")
axes[2].set_xlabel("GrLivArea"); axes[2].legend(fontsize=8)

plt.suptitle("Comparacion de metodos de deteccion de outliers", fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Que hacer con los outliers

Detectar un outlier es solo el primer paso. La accion correcta depende del **contexto**:

| Estrategia | Cuando usarla | Codigo |
|---|---|---|
| **Conservar** | Son observaciones legitimas importantes | No hacer nada |
| **Eliminar** | Son errores de medicion o entrada | `df.drop(outlier_idx)` |
| **Winsorizar** | Existen pero distorsionan; se limita el rango | `np.clip(x, lower, upper)` |
| **Transformar** | Sesgo por colas; log reduce la escala | `np.log1p(x)` |
| **Indicador binario** | El hecho de ser outlier es informativo | `df['flag'] = (x > umbral)` |

> **Regla practica:** en Ames Housing, las casas >4000 sqft son outliers *reales*
> pero su comportamiento de precio es diferente al del mercado tipico. El paper
> original de De Cock (2011) recomienda eliminarlas para predicciones en el rango
> normal del mercado.

In [ ]:
# --- Estrategia 1: Winsorizar (Winsorizing / Capping) ---
# Winsorizar = limitar los valores al rango [percentil_inf, percentil_sup].
# No elimina filas; solo recorta los extremos al valor mas cercano dentro del rango.

area_orig = df["GrLivArea"].dropna().copy()

# Calculamos los percentiles de recorte (5% y 95%).
p05 = area_orig.quantile(0.05)  # limite inferior: el 5% mas pequeno se recorta aqui
p95 = area_orig.quantile(0.95)  # limite superior: el 5% mas grande se recorta aqui

# np.clip limita cada valor al rango [p05, p95].
# Valores menores que p05 quedan en p05; mayores que p95 quedan en p95.
area_winsored = area_orig.clip(lower=p05, upper=p95)

print("Winsorizing (percentiles 5%-95%)")
print(f"  Antes: min={area_orig.min():.0f}  max={area_orig.max():.0f}  std={area_orig.std():.0f}")
print(f"  Despues: min={area_winsored.min():.0f}  max={area_winsored.max():.0f}  std={area_winsored.std():.0f}")
print(f"  Filas modificadas: {(area_orig != area_winsored).sum()}")

In [ ]:
# --- Estrategia 2: Eliminar outliers ---
# Solo si estamos seguros de que son errores o casos fuera del alcance del modelo.
# En Ames, las casas >4000 sqft son legitimas pero atipicas para el mercado general.

# Definimos el umbral segun el criterio de De Cock (2011) y el analisis de Ames.
umbral_area = 4000

# Creamos una mascara de filas a conservar (True = conservar).
mask_normal = df["GrLivArea"] <= umbral_area

# Aplicamos la mascara para filtrar.
df_clean = df[mask_normal].copy()

print(f"Eliminar GrLivArea > {umbral_area} sqft")
print(f"  Dataset original: {len(df)} filas")
print(f"  Dataset limpio:   {len(df_clean)} filas")
print(f"  Eliminadas:       {len(df) - len(df_clean)} casas")

In [ ]:
# --- Estrategia 3: Transformar (log1p) ---
# log1p comprime las colas sin eliminar puntos.
# Es especialmente util cuando los outliers son reales pero distorsionan el modelo.

# log1p(x) = log(1 + x): maneja x=0 sin error (log(0) = -inf).
area_log = np.log1p(area_orig)

# Calculamos el skewness antes y despues.
def skewness(a):
    a = np.asarray(a)
    return float(((a - a.mean()) ** 3).mean() / a.std() ** 3)

print("Transformacion log1p")
print(f"  GrLivArea original:  skew={skewness(area_orig):+.3f}  std={area_orig.std():.0f}")
print(f"  log1p(GrLivArea):    skew={skewness(area_log):+.3f}  std={area_log.std():.3f}")
print(f"  La inversa es expm1: expm1(log1p(x)) = x")

In [ ]:
# --- Estrategia 4: Variable indicadora (flag binario) ---
# A veces el hecho de SER outlier es informativo para el modelo.
# Ejemplo: las casas muy grandes (outliers en GrLivArea) pueden tener un
# comportamiento de precio distinto que el modelo deberia conocer.

# Creamos una columna binaria: 1 si la casa es un outlier de area, 0 si no.
df_flagged = df.copy()
df_flagged["is_large_house"] = (df_flagged["GrLivArea"] > umbral_area).astype(int)

n_large = df_flagged["is_large_house"].sum()
print(f"Variable indicadora 'is_large_house' (GrLivArea > {umbral_area})")
print(f"  Casas grandes (flag=1): {n_large}")
print(f"  Casas normales (flag=0): {len(df_flagged) - n_large}")

# Verificamos que el precio promedio difiere entre grupos.
if "SalePrice" in df_flagged.columns:
    precio_por_grupo = df_flagged.groupby("is_large_house")["SalePrice"].agg(["mean", "median", "count"])
    print("\nPrecio segun el flag:")
    print(precio_por_grupo)

## Repaso

### Metodos de deteccion

| Metodo | Fortaleza | Limitacion |
|---|---|---|
| **z-score** | Simple, rapido | Sensible a outliers (usa media/std) |
| **IQR / Tukey** | Robusto (usa Q1/Q3) | Solo univariante |
| **MAD** | Mas robusto (usa mediana) | Solo univariante |
| **Isolation Forest** | Multivariante, no asume distribucion | Necesita calibrar contamination |
| **DBSCAN** | Detecta clusters arbitrarios + ruido | Sensible a eps y min_samples |

### Estrategias de tratamiento

1. **Conservar** — si son observaciones legitimas e importantes para el modelo.
2. **Eliminar** — si son errores de medicion o estan fuera del alcance de prediccion.
3. **Winsorizar** — si existen pero distorsionan; se recortan al percentil 95/5.
4. **Transformar** — `log1p` reduce el sesgo y comprime las colas sin eliminar datos.
5. **Indicador binario** — cuando la condicion de outlier es informativa para el precio.

> **Conexion con los otros notebooks:**  
> Los outliers detectados aqui afectan directamente las decisiones de imputacion
> (Notebook 1: usar mediana en vez de media) y de escalado (Notebook 1: RobustScaler
> en vez de StandardScaler). Las features limpias van al feature store (Notebook 3)
> y alimentan el modelo de regresion (Notebook 4).